# 2D latent GAN sample

Generate categorical 2D images with the trained WGAN generator and inspect the fixed critic scores.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.app.runtime import (
    build_dataset,
    build_loader,
    load_defaults,
    load_run_critic,
    load_run_generator,
    load_run_vae,
)
from src.modeling.phases import probabilities_to_labels

## Parameters

In [ ]:
RUN_DIR = None
BATCH_SIZE = 4
PHASE_FRACTIONS = None  # Example: [0.3, 0.1, 0.6]

## Generate

In [ ]:
run_dir = Path(RUN_DIR) if RUN_DIR else max(
    [p for p in (ROOT / "run").glob("*") if (p / "gan.yaml").is_file()],
    key=lambda p: p.stat().st_mtime,
)
run_dir = run_dir if run_dir.is_absolute() else ROOT / run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args = Namespace(**load_defaults(run_dir / "gan.yaml"))
args.data_dir = Path(args.data_dir)
args.data_dir = args.data_dir if args.data_dir.is_absolute() else ROOT / args.data_dir
args.batch_size = BATCH_SIZE

dataset = build_dataset(args)
real_images = next(build_loader(dataset, args, device=torch.device("cpu"))).to(device)
vae = load_run_vae(run_dir, device)
generator = load_run_generator(run_dir, device)
critic = load_run_critic(run_dir, device)

if PHASE_FRACTIONS is None:
    real_labels = real_images[:, 0].long()
    fractions = F.one_hot(real_labels, num_classes=args.num_phases).float().mean(dim=(1, 2))
else:
    fractions = torch.tensor(PHASE_FRACTIONS, device=device, dtype=torch.float32)
    fractions = fractions.unsqueeze(0).expand(BATCH_SIZE, -1)

with torch.no_grad():
    real_latent, _ = vae.encode(real_images)
    noise = torch.randn(BATCH_SIZE, args.noise_ch, device=device)
    fake_latent = generator(noise, fractions)
    probabilities = vae.decode_probs(fake_latent)
    generated = probabilities_to_labels(probabilities, args.num_phases)
    real_scores = critic(real_latent, F.one_hot(real_images[:, 0].long(), num_classes=args.num_phases).float().mean(dim=(1, 2)))
    fake_scores = critic(fake_latent, fractions)

generated_fraction = F.one_hot(generated[:, 0], num_classes=args.num_phases).float().mean(dim=(0, 1, 2))
print("run:", run_dir)
print("device:", device)
print("target fraction:", [round(value, 4) for value in fractions.mean(dim=0).tolist()])
print("generated fraction:", [round(value, 4) for value in generated_fraction.tolist()])
print(f"critic real: {real_scores.mean().item():.4f}")
print(f"critic generated: {fake_scores.mean().item():.4f}")

## Check

In [ ]:
count = min(BATCH_SIZE, 4)
fig, axes = plt.subplots(2, count, figsize=(3 * count, 6), squeeze=False)
for index in range(count):
    axes[0, index].imshow(real_images[index, 0].cpu(), cmap="gray", vmin=0, vmax=args.num_phases - 1, interpolation="nearest")
    axes[0, index].set_title("real")
    axes[1, index].imshow(generated[index, 0].cpu(), cmap="gray", vmin=0, vmax=args.num_phases - 1, interpolation="nearest")
    axes[1, index].set_title("generated")
for axis in axes.ravel():
    axis.axis("off")
plt.tight_layout()